In [11]:
import pandas as pd
from pathlib import Path
import re

## Muestra de proyectos PSPLIB
Para este trabajo de comparación de modelos

(1) De cada subset de PSPLIB original {J120,J90,J60,J30} se escogen 100 aleatorios, total 400

In [135]:
# Muestra de 100 proyectos de cada subset (400 en total)
# carpeta con los archivos .sm
df = pd.read_csv("./PSPLIB_sample/psplib_sample_combined.csv")
print('Number of files:', len(df))
results_dir = Path("./PSPLIB_sample/")
for file in results_dir.glob("*.sm"):
    if file.name not in df["FileName"].values:
        print(f"Archivo {file.name} no encontrado en el CSV")


Number of files: 400


(2) De cada 100 proyectos de subset, se escogen 10% para selección de hiperparámetros de cada faimila de modelos y el resto 90% para la comparación de modelos

In [137]:
df.groupby(['SubSet','model_selection']).count()['FileName']

SubSet  model_selection
J120    False              90
        True               10
J30     False              90
        True               10
J60     False              90
        True               10
J90     False              90
        True               10
Name: FileName, dtype: int64

## Selección de modelos

```
Para cada configuración LHS:
    Para cada proyecto de model_selection:
        entrenar con 80% de replicaciones
        evaluar con 20% fijo
    agregar errores
elegir configuración con mejor rendimiento medio/ranking
```

In [138]:
# Parametrización
# Número de configuraciones LHS: 50
for file in Path("model_selection/hyperparameters").glob("*.csv"):
    df_temp = pd.read_csv(file)
    print(f"{file.name}: {len(df_temp)}")

lhs_ebm_parameters.csv: 50
lhs_en_parameters.csv: 50
lhs_fastai_parameters.csv: 50
lhs_rf_parameters.csv: 50
lhs_sv_parameters.csv: 50
lhs_xgb_parameters.csv: 50


In [139]:
# Proyectos seleccionados para cada subset
# Número de proyectos para la selección de hiperparámetros de modelos: 
#   40 = 10 x {J120,J90,J60,J30}  
# (los IDs están en /model_selection/montecarlo/project_selection.csv)
df_mselect = pd.read_csv("./model_selection/project_selection.csv")
df_mselect.groupby("SubSet").count()['sample']

SubSet
J120    10
J30     10
J60     10
J90     10
Name: sample, dtype: int64

In [144]:
# Número de replicaciones de montecarlo: 10K
df=pd.DataFrame({'files': list(Path("montecarlo").glob("*.csv"))})
for _ in range(5):
    f= df.sample(n=1)['files'].iloc[0]
    print(f.name,pd.read_csv(f).shape)


j3045_9_simulation_01.csv (10000, 35)
j12012_3_simulation_01.csv (10000, 125)
j9025_9_simulation_01.csv (10000, 95)
j3016_5_simulation_01.csv (10000, 35)
j607_8_simulation_01.csv (10000, 65)


In [145]:
# Número de runs para test (20% de 10K)
# (los IDs están en el archivo /model_selection/montecarlo/test_runs.csv)
print (pd.read_csv("montecarlo/test_runs.csv").shape)

(2000, 1)


## Evaluación de modelos

In [143]:
# Numero de proyectos para la evaluación de modelos: 360 (90 x {J120,J90,J60,J30})
df_meval = pd.read_csv("./model_evaluation/project_evaluation.csv")
df_meval.groupby("SubSet").count()['sample']

SubSet
J120    90
J30     90
J60     90
J90     90
Name: sample, dtype: int64